# Modelling with AI partners {#sec-ai-partners}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/c17-ai-partners.ipynb)

A language model can read forty interviews in a minute, draft a causal loop diagram, write model equations, propose tests and summarise a policy run for a board. It can also invent a quote, reverse a polarity, misstate a number and agree with whatever the question implies. This chapter sets up how the rest of the book works with such models: what they are, the three ways we use them, and the checks that make their help safe to rely on.

```
              asks                      calls
  Modeller ---------> Language model ---------> Tools (Python:
     ^                      |                    simulate, test,
     |                draft, critique            fit, check units)
     |                 or plan                        |
     |                      v                         |
     +------------ Checks by code <------------------+
                                       numbers
```

*How the book uses AI: the model drafts, critiques or plans; Python tools compute every number; code checks the result before a person accepts it.*

The book is organised by the stages of a modelling project: conceptualise the problem and a hypothesis, formulate the model, test and calibrate it, and design and evaluate policy [@richardson1981; @sterman2000; @martinezmoyano2013]. Each stage ends with a chapter on AI at that stage. This chapter is the common ground they build on. It uses the waiting-list model of @sec-push-back and introduces no molecules.

**Who it is for.** Everyone. **Who can skip.** Readers who already build with language models can skim sections 1 and 2, but should read sections 3, 5 and 6, which set the rules every later AI section follows. **Prerequisites.** @sec-push-back. **Time.** About two hours.

**Learning outcomes.** By the end of this chapter you can:

1. **explain** what a language model does and why its output must be checked (section 1, Try it);
2. **choose** between generative, reasoning and agent uses for a modelling task (section 2, Try it);
3. **record** and replay a model's reply, and **check** a structured reply by code (section 3, Try it);
4. **test** a reasoning model's argument by simulation (section 4, Try it);
5. **build** an agent with a simulation tool, and **verify** that every number in its answer came from a tool (section 5, Try it);
6. **apply** the book's contract for AI use, including what to record and what stays human (section 6, Try it).

In [ ]:
# Run this cell first. In Colab it installs loopbricks (about ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
from loopbricks import ai
import copy
import json
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 2
rng = np.random.default_rng(SEED)

# Recorded replies: the local file if there is one, otherwise GitHub's copy
REPLAY = "data/ai/c17.json" if os.path.exists("data/ai/c17.json") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/book-ai/notebooks/data/ai/c17.json"

OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00",
             "#0072B2", "#CC79A7"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = cycler(color=OKABE_ITO) + \
    cycler(linestyle=STYLES)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})

# The waiting list of chapter 1: one stock, and referrals that fall as
# waits grow (sensitivity 0.5); Extra capacity starts in month 12.
case_spec = {
    "name": "Waiting list", "timeUnit": "month", "T": 119, "dt": 0.125,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": "32000",
                "units": "patients", "nonNegative": True}],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl",
         "eq": "Referral_rate * (Waiting_time / Normal_waiting_time)"
               " ^ (-Referral_sensitivity)", "units": "patients/month"},
        {"id": "trt", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
         "units": "patients/month"},
    ],
    "vars": [
        {"id": "rr", "name": "Referral rate", "eq": "4000",
         "units": "patients/month"},
        {"id": "cap", "name": "Capacity",
         "eq": "3950 + STEP(Extra_capacity, 12)", "units": "patients/month"},
        {"id": "mtt", "name": "Minimum treatment time", "eq": "0.5",
         "units": "month"},
        {"id": "extra", "name": "Extra capacity", "eq": "0",
         "units": "patients/month"},
        {"id": "wt", "name": "Waiting time", "eq": "Waiting_list / Treatments",
         "units": "month"},
        {"id": "nwt", "name": "Normal waiting time", "eq": "8",
         "units": "month"},
        {"id": "sens", "name": "Referral sensitivity", "eq": "0.5",
         "units": "dimensionless"},
    ],
}
looped = lb.Model.from_spec(case_spec)
print("loopbricks", lb.__version__, "| replies from", REPLAY)

## 1. What a language model is, and is not

A **large language model** (LLM) is a program trained on a very large body of text to continue text: given everything so far, it predicts a likely next piece, adds it, and repeats. Training on code, papers and reports gives it a wide, shallow knowledge of system dynamics, statistics and Python, and the ability to follow instructions written in plain language. **Generative AI** is the wider family of models that produce text, code or images this way.

Three properties matter for a modeller.

- **Fluent is not true.** The model produces what is likely, not what is checked. A confident, well-formed answer can contain a quote nobody said, a reference that does not exist or a number nobody computed. This is called **hallucination** [@ji2023hallucination].
- **It sees only what you send.** The model knows nothing about your project beyond the **prompt**, the text sent to it, and keeps no memory between requests unless the earlier messages are sent again.
- **The same question can get different answers.** Replies are sampled, and models are updated. A reply is a source with a date, like an interview, not a fixed fact.

The last point is a problem for a book whose rule is that every number comes from running the code. Our answer is the **replay file**: every request a chapter makes, and the reply it got, is stored in a file next to the notebook. A chapter then runs anywhere, offline and free, and gives the same text every time. The `loopbricks.ai` module does this. Its key for a request is a hash of everything sent: the model's name, the instructions, the messages and any tool results. Change one character and the key changes, so a stale reply can never be passed off as the answer to a new question.

In [ ]:
a = ai.request_key([{"role": "user", "content": "List the stocks."}])
b = ai.request_key([{"role": "user", "content": "List the stocks. "}])
print(a, b, "same" if a == b else "different")
print("default model:", ai.MODEL)

Adding one space gives a different key. The default model name is `claude-opus-5-5`; set the environment variable `LB_AI_MODEL` to use another.

**Where the replies in this book come from.** No chapter calls a model service while it runs. The replies in the replay files were written for the book, with the help of the model named above, and each record says so in its `how` field. They are realistic, and some contain faults planted on purpose, which the text points out. To ask a live model instead, install the client (`pip install anthropic`), set `ANTHROPIC_API_KEY` to your key and `LB_AI_LIVE=1`, and run the chapter: requests that have a recorded reply still replay, and any new request goes to the live model, whose reply is added to the file with `how` set to "live call".

**Try it 1.**

- *Check.* Why does the book replay recorded replies instead of calling a model each time it runs?
- *Apply.* Two requests differ only in the model name. Use `ai.request_key(messages, model=...)` to show that they get different keys. Why is that the right behaviour?
- *Extend.* A colleague edits a prompt in a chapter and the notebook stops with `MissingReply`. List the two ways to make it run again, and say which one keeps the rule that every reply is honestly labelled.

<details><summary>Worked answer</summary>

*Check.* So that every reader gets the same text, offline and free, and so that the numbers and checks in the prose stay true. A live reply can change from one run to the next.

*Apply.* The cell below shows the keys differ. A different model can answer differently, so its reply must not be replayed as if it came from the first one.

*Extend.* Either run live (`LB_AI_LIVE=1` with a key), which records the new reply as a live call, or write a new reply by hand with `ai.record` or `ai.author`, giving a `how` that says it was written. Both keep the label honest; what is never allowed is copying an old reply under the new key.

</details>

In [ ]:
msgs = [{"role": "user", "content": "List the stocks."}]
print(ai.request_key(msgs, model="claude-opus-5-5"),
      ai.request_key(msgs, model="another-model"))

## 2. Three kinds of partner

We use language models in three ways, and choosing the right one is half the skill.

| Kind | What it does | Good for | Its typical failure | The check |
|---|---|---|---|---|
| **Generative** | Drafts text, tables, code or a model spec from instructions | First drafts: variable lists, diagrams, equations, briefs | Fluent, plausible and wrong | Code that tests the draft: quotes found, spec loads, units balance |
| **Reasoning** | Works through a problem step by step before it answers | Critique: is this polarity right, what does this result rest on, why did this test fail | A long, convincing argument with one wrong step | Simulate the claim the argument makes |
| **Agent** | Chooses tools (Python functions we write) in a loop and reads their results | Search and iteration: find a policy, repair a spec, hunt for a failing test | Stops early, wanders, or reports a number no tool gave | Rerun its answer; every number traced to a tool result |

A **reasoning model** is a language model trained or prompted to work through intermediate steps before it answers, which helps on problems with several steps [@wei2022cot]. In `loopbricks.ai`, `reasoning=True` asks for this. An **agent** is a model that alternates between reasoning about what to do and acting through tools, then reads what the tools return [@yao2022react]. The tools are ordinary Python functions; the model only chooses which to call and with what arguments. That division is what makes agents useful to a modeller: the simulation computes every number, and the model does the searching.

All three appear at every stage of the book. Two reviews of language models in system dynamics are worth reading early: @akhavan2024 on how to use them and how not to, and @hosseinichimeh2024 on drafting causal loop diagrams from text.

**Try it 2.**

- *Check.* A manager wants a one-paragraph summary of a policy run. Which kind of partner fits, and what check would you run on its output?
- *Apply.* For each task, choose a kind: (a) list rival explanations for a reference mode; (b) find the smallest budget that meets a target; (c) turn a table of equations into a plain-language description.
- *Extend.* Why should an agent never be given a tool that returns made-up numbers, such as "estimate the waiting time from your knowledge"?

<details><summary>Worked answer</summary>

*Check.* Generative. Check that every number in the summary appears in the run results, and that every variable it names exists in the model.

*Apply.* (a) Reasoning, then code that tests each rival against the data. (b) An agent with a simulation tool, or plain code if the search is simple. (c) Generative, with a check that every name it uses is in the model.

*Extend.* Because the whole value of an agent is that its numbers come from computation. A tool that asks a model for a number launders a guess into a result, and the trace check of section 5 could no longer tell them apart.

</details>

In [ ]:
choice = {"(a) rival explanations": "reasoning",
          "(b) smallest budget for a target": "agent",
          "(c) equations to plain language": "generative"}
for task, kind in choice.items():
    print(f"{task:<36} {kind}")

## 3. Asking, recording and replaying

The simplest use is generative: ask for a draft and check it. Here we give the model the story of chapter 1 in four sentences and ask for the stocks, flows and auxiliaries, with units, as JSON. **JSON** is a text format for lists and dictionaries that both models and Python read easily; `ai.ask_json` asks and then reads the first JSON object in the reply.

In [ ]:
STORY = ("A hospital group's outpatient list holds about 32,000 patients. "
         "GPs refer about 4,000 patients a month, and clinics can treat "
         "about 3,950 a month. Nobody is seen sooner than half a month "
         "after referral. When waits get long, some GPs stop referring.")

PROMPT = f"""You are helping a system dynamics modeller.
From the story below, list the stocks, flows and auxiliary variables.
Give each a name that is a noun, neutral in direction, and a unit.
Units of time are months. For each flow, say which stock it fills
("to") or drains ("from").
Return only JSON:
{{"stocks": [{{"name": ..., "units": ...}}],
 "flows": [{{"name": ..., "units": ..., "to" or "from": ...}}],
 "auxiliaries": [{{"name": ..., "units": ...}}]}}

STORY: {STORY}"""

draft = ai.ask_json(PROMPT, replay=REPLAY)
print(json.dumps(draft, indent=1))

The draft names one stock, two flows and four auxiliaries. Now the check, which is the part that matters. A flow must be measured in its stock's units per unit of time, and must fill or drain a stock that exists. Code tests both, and compares the names with the model we built by hand in @sec-push-back.

In [ ]:
def check_draft(d, time_unit="month"):
    """Problems in a drafted list of stocks and flows (empty if none)."""
    stocks = {s["name"]: s["units"] for s in d["stocks"]}
    problems = []
    for f in d["flows"]:
        stock = f.get("to") or f.get("from")
        if stock not in stocks:
            problems.append(f"{f['name']}: no stock called {stock!r}")
        elif f["units"] != f"{stocks[stock]}/{time_unit}":
            problems.append(f"{f['name']}: units {f['units']!r}, "
                            f"expected {stocks[stock]}/{time_unit}")
    return problems


print("problems:", check_draft(draft) or "none")
names = {x["name"] for k in ("stocks", "flows", "auxiliaries")
         for x in draft[k]}
ours = {x["name"] for k in ("stocks", "flows", "vars") for x in case_spec[k]}
print("in both:", sorted(names & ours))   # & : names in both sets
print("only in the draft:", sorted(names - ours))
print("only in our model:", sorted(ours - names))

The units and the stock references pass. Six names match chapter 1's model exactly, and the draft calls the 4,000 a month `Normal referrals` where we wrote `Referral rate`. What the draft lacks is the two parameters that say *how* GPs respond to waits, `Normal waiting time` and `Referral sensitivity`, because the story does not give them (`Extra capacity` is our policy lever, not part of the story). Those gaps are exactly the questions a modeller must take back to the people who know the system.

Each reply is stored with a note of how it was made and when. The record is part of the project's evidence, like an interview transcript.

In [ ]:
records = ai.records(REPLAY)
key = ai.request_key([{"role": "user", "content": PROMPT}])
print("key:", key)
print("how:", records[key]["how"])
print("date:", records[key]["date"])
try:
    ai.ask(PROMPT + " Be brief.", replay=REPLAY)
except ai.MissingReply:
    print("a changed prompt has no recorded reply: MissingReply")

**Try it 3.**

- *Check.* What would `check_draft` report if the draft gave `Treatments` the units `patients`?
- *Apply.* Write a check that every stock and flow name in the draft, in lower case, has at least one word that appears in the story. Which names fail, and is that a fault of the draft?
- *Extend.* Make the units check accept `patients per month` as well as `patients/month`. Why is it better to normalise units in the check than to ask the model to be consistent?

<details><summary>Worked answer</summary>

*Check.* `Treatments: units 'patients', expected patients/month`: the cell below breaks a copy of the draft and shows it.

*Apply.* The cell below finds the names with no word in the story. `Referrals` and `Treatments` fail only because the story says "refer" and "treat"; this check is crude, and a failing name is a prompt to look, not proof of invention.

*Extend.* Normalise by replacing " per " with "/" before comparing. Models vary their wording even when told not to; a check that tolerates harmless variation and still catches real errors is more robust than a request.

</details>

In [ ]:
broken = copy.deepcopy(draft)
for f in broken["flows"]:
    if f["name"] == "Treatments":
        f["units"] = "patients"
print(check_draft(broken))

story_words = set(re.findall(r"[a-z]+", STORY.lower()))
for k in ("stocks", "flows"):
    for x in draft[k]:
        words = set(re.findall(r"[a-z]+", x["name"].lower()))
        if not words & story_words:
            print("no word from the story:", x["name"])

## 4. A reasoning model as critic

Chapter 1 found that adding 400 treatments a month shortens the list by only about 3,000 patients, because shorter waits draw back referrals. A good critic asks what that result rests on. We send the model's equations and the result to a reasoning model and ask it to name the assumption that matters most, explain why, and propose a test as JSON we can run.

In [ ]:
base = looped.run(extra_capacity=0).loc[119]
more = looped.run(extra_capacity=400).loc[119]
RESULT = (f"With 400 extra treatments a month from month 12, the list at "
          f"month 119 is {more['Waiting list']:,.0f} against "
          f"{base['Waiting list']:,.0f} without them.")

CRITIQUE = f"""You are reviewing a system dynamics model before it is
shown to a hospital board. Here are its equations:

{looped.equations()}

Result: {RESULT}

Which single assumption does this result depend on most? Explain why,
using the equations. Then propose one test that could change the
conclusion: a parameter to vary and the values to try.
Return only JSON: {{"assumption": ..., "why": ..., "prediction": ...,
"test": {{"parameter": <name as in the equations>, "values": [...]}}}}"""

critique = ai.ask_json(CRITIQUE, replay=REPLAY, reasoning=True)
for k in ("assumption", "why", "prediction"):
    print(f"{k}: {critique[k]}\n")
print("test:", critique["test"])

The critic names the referral sensitivity, and its reasoning goes further than chapter 1 did: in equilibrium, referrals equal treatments, so the list is $L^{*} = 8\,C\,(4000/C)^{1/s}$, where $C$ is capacity and $s$ the sensitivity. At $s = 1$ the capacity cancels, and the list does not move at all; above 1, extra capacity makes it longer. That is a strong claim, and the reasoning is only as good as each step. So we do not accept it; we run the test it proposed, and check the formula against the simulation.

In [ ]:
param = lb.name_key(critique["test"]["parameter"])
assert param in looped.params, f"no parameter {param}"
rows = []
for s in critique["test"]["values"]:
    lists = [looped.run(extra_capacity=x, **{param: s}).loc[119,
                                                              "Waiting list"]
             for x in (0, 400)]
    formula = 8 * 4350 * (4000 / 4350) ** (1 / s) if s > 0 else np.nan
    rows.append((s, *lists, lists[0] - lists[1], formula))
sweep = pd.DataFrame(rows, columns=["sensitivity", "list, no extra",
                                    "list, +400", "shorter by",
                                    "formula, +400"])
print(sweep.to_string(index=False, float_format=lambda v: f"{v:,.0f}",
                      formatters={"sensitivity": "{:.2f}".format}))

The simulation bears the critic out. With fixed referrals ($s = 0$) the extra capacity empties most of the list: it ends 35,950 patients shorter. At the chapter's value of 0.5 it is 2,978 shorter. At 1 the list is 32,000 with or without the extra capacity, and at 2 the extra capacity leaves it 1,571 *longer*. The formula matches the simulated list within a few hundred patients where the model has settled (24,881 against 24,963 at 0.25, and 29,425 against 29,426 at 0.5). The test the critic proposed is now the most important thing to measure before the board meets: the sign of the advice depends on it.

Note the division of labour. The reasoning model did something useful that is easy to skip: it read the equations and derived a testable consequence. The simulation did the deciding. A reasoning model that skipped a step or slipped a sign would have been caught by the same table.

**Try it 4.**

- *Check.* From the formula, what list does $s = 1$ give, whatever the capacity, and why?
- *Apply.* Run the sweep for $s = 1.5$ and compare the simulated list with the formula.
- *Extend.* At $s = 1$ the list does not move, but the board cares about the wait. What does the extra capacity do to the waiting time at month 119 when $s = 1$?

<details><summary>Worked answer</summary>

*Check.* $8 \times 4000 = 32{,}000$ patients. With $s = 1$, $L^{*} = 8\,C \times 4000 / C$, so capacity cancels: every extra treatment draws back one extra referral.

*Apply.* The cell below runs it. The list with 400 extra is longer than without, as the formula predicts for any $s$ above 1.

*Extend.* The wait falls, because the same list is now treated faster: the cell below shows it drops from about 8.1 to about 7.4 months. The patients are better off even though the list is not shorter, which is chapter 1's advice to judge a policy by waits, not by the list.

</details>

In [ ]:
s = 1.5
sim = looped.run(extra_capacity=400, referral_sensitivity=s).loc[
    119, "Waiting list"]
no_extra = looped.run(referral_sensitivity=s).loc[119, "Waiting list"]
print(f"s = {s}: simulated {sim:,.0f}, formula "
      f"{8 * 4350 * (4000 / 4350) ** (1 / s):,.0f}, no extra {no_extra:,.0f}")
for x in (0, 400):
    wt = looped.run(extra_capacity=x, referral_sensitivity=1).loc[
        119, "Waiting time"]
    print(f"s = 1, extra {x}: waiting time {wt:.2f} months")

## 5. A first agent

The board's question is a search: *how much extra capacity brings the average wait below six months?* An agent fits. We give the model one tool, a Python function that runs the model, and a task. The `ai.agent` loop sends the task and a description of the tool; the model replies with a tool call as JSON; the loop runs the tool here, in Python, and sends back the result; and so on until the model gives an answer. The tool's signature and docstring are all the model sees of it, so the docstring must say exactly what goes in and what comes out.

In [ ]:
def simulate(extra_capacity: float) -> dict:
    """Run the waiting-list model (referrals respond to waits, sensitivity
    0.5) with extra_capacity treatments a month from month 12. Returns the
    waiting time (months), waiting list (patients) and treatments
    (patients a month) at month 119."""
    r = looped.run(extra_capacity=extra_capacity).loc[119]
    return {"waiting_time": round(float(r["Waiting time"]), 2),
            "waiting_list": round(float(r["Waiting list"])),
            "treatments": round(float(r["Treatments"]))}


TASK = ("The board wants the average waiting time at month 119 below 6 "
        "months. Find, to the nearest 25 patients a month, the smallest "
        "extra capacity that achieves it. Report that capacity with its "
        "waiting time and list.")
steps = ai.agent(TASK, {"simulate": simulate}, replay=REPLAY)
ai.show(steps)

The agent made six calls. It started from the 400 a month that chapter 1 discussed, doubled to 800 to bracket the target, and then halved the interval until the step was 25 a month: 600, 700, 650 and 675. Its answer is 675 extra treatments a month, giving a wait of 5.98 months and a list of 27,676, and it notes that 650 is not enough.

An agent's answer is a claim. Three checks turn it into a result. First, every number in the answer must appear in a tool result, in an argument the agent chose, or in what the agent was told (the task and the tool's docstring); anything else was not computed. Second, rerunning the tool with the answer must give the same numbers. Third, where the search is cheap, plain code should find the same answer.

In [ ]:
def numbers(text):
    """Every number written in a text, as floats ("27,676" is 27676)."""
    return {float(x.replace(",", ""))
            for x in re.findall(r"\d[\d,]*(?:\.\d+)?", str(text))}


def unsupported(answer, steps, allowed=""):
    """Numbers in an answer that no tool result, tool argument or the
    allowed text contains (within 1%, so a rounded figure passes)."""
    seen = numbers(allowed)
    for s in steps:
        seen |= numbers(s.get("result", "")) | numbers(s.get("args", ""))
    return sorted(n for n in numbers(answer)
                  if not any(abs(n - m) <= 0.01 * max(abs(m), 1)
                             for m in seen))


answer = steps[-1]["answer"]
told = TASK + simulate.__doc__          # what the agent was told
print("unsupported numbers:", unsupported(answer, steps, told) or "none")
print("rerun 675:", simulate(675), "| 650:", simulate(650))



def threshold(target, sens=0.5):
    """Smallest extra capacity (to 1 a month) with the wait below target,
    by bisection: halve the interval that contains the answer."""
    lo, hi = 0.0, 4000.0
    while hi - lo > 1:
        mid = (lo + hi) / 2
        wt = looped.run(extra_capacity=mid,
                        referral_sensitivity=sens).loc[119, "Waiting time"]
        lo, hi = (lo, mid) if wt < target else (mid, hi)
    return hi


print(f"code finds the threshold at about {threshold(6):.0f} a month")

All three checks pass. Every number in the answer traces to a tool result or the task, the rerun gives the same wait and list, and bisection in code puts the exact threshold at about 669 a month, so 675 is indeed the smallest multiple of 25 that meets the target. For a one-variable search, the code was simpler than the agent. Agents earn their place when the search needs judgement: several levers, tools of different kinds, results that must be read before choosing the next step. Chapters 10, 14 and 16 give them such work.

**Try it 5.**

- *Check.* How many tool calls did the agent make, and which result told it that 650 was not enough?
- *Apply.* Use the bisection code to find the threshold to 1 patient a month when the board's target is 5.5 months.
- *Extend.* Repeat the search with the referral sensitivity at 0.25 and at 1 (add a `referral_sensitivity` argument to a copy of `simulate`). How much does the answer depend on the assumption section 4 found?

<details><summary>Worked answer</summary>

*Check.* Six calls; the fifth, `simulate(650)`, returned a wait of 6.05 months.

*Apply* and *Extend.* The cell below runs both. The answer to the board's question depends heavily on the referral sensitivity: the less GPs respond to waits, the less extra capacity is needed to cut them: about 353 a month at a sensitivity of 0.25, 669 at 0.5 and 1,384 at 1, more than twice as much. A target of 5.5 months needs about 875 a month. That is the critic's point in section 4, now in the board's units.

</details>

In [ ]:
print("steps with a tool call:", sum("tool" in s for s in steps))
print(f"target 5.5 months: about {threshold(5.5):.0f} a month")
for s in (0.25, 0.5, 1):
    print(f"sensitivity {s}: about {threshold(6, s):.0f} a month")

## 6. The contract

Every AI section in the book follows five rules. They are the reason the help can be trusted, and they apply equally to your own projects.

1. **The modeller owns the model.** AI drafts, critiques and searches; a person decides, and the record says who.
2. **Check by code, not by eye.** Every AI output meets a test: quotes found in the source, a spec that loads, units that balance, numbers that trace to a run. Reading a fluent answer and finding it convincing is not a check.
3. **Numbers come from tools.** An agent may only report numbers that a tool returned. The `unsupported` check of section 5 enforces it.
4. **Record what was asked.** Keep the prompt, the model, the date and the replay key with every output, as you would cite any other source. The replay file does this for you.
5. **Protect people's data.** Do not send confidential interviews, patient records or commercial data to an external service without consent and a data agreement. A model that runs on your own machine is one option; an anonymised summary is another.

What stays human is the part no check can do: deciding what the problem is and whose problem it is, judging whether a quote really supports a link, choosing which uncertainty matters to the decision, and taking responsibility for advice. Two reviews make the same point from experience with system dynamics projects [@akhavan2024; @martinezmoyano2013].

**Try it 6.**

- *Check.* Which rule does a brief break if it says "the model shows" about a number a language model wrote?
- *Apply.* Build a table of every record in this chapter's replay file: its key, how it was made, its date and the first 50 characters of the reply.
- *Extend.* Your organisation forbids sending interview notes outside. Which parts of this chapter's workflow still work, and how?

<details><summary>Worked answer</summary>

*Check.* Rule 3, numbers come from tools, and so rule 2: no check traced the number to a run.

*Apply.* The cell below builds the table from the file. Every record says it was written for the book; a reader who runs the chapter live will see new records marked as live calls.

*Extend.* Everything that does not send the notes: the checks, the tools and the replay files are local. Use a model that runs on your own computer, or send only an anonymised summary that the data owner has approved; record which you did.

</details>

In [ ]:
log = pd.DataFrame([(k, r["how"], r["date"], " ".join(r["text"].split())[:50])
                    for k, r in ai.records(REPLAY).items()],
                   columns=["key", "how", "date", "reply starts"])
print(log.to_string(index=False))

## 7. AI at every stage of this book

Each stage of the modelling process ends with a chapter on AI at that stage. They follow the pattern of this chapter: the task, the kind of partner, the prompt or tools, the replayed reply, the check by code, and what stays human.

| Stage | Chapter | Generative | Reasoning | Agent |
|---|---|---|---|---|
| 1. Conceptualise | @sec-ai-conceptualise | Variables and a causal loop diagram from interviews, with quotes | Critique of the problem statement and reference mode | Rival hypotheses tested against the data |
| 2. Formulate | @sec-ai-formulate | A model spec from the diagram; documentation | Review of decision rules and units | Write, check and run a spec until it passes |
| 3. Test | @sec-ai-test | Test plans written as code | Diagnosing why a test failed | Hunting for failing tests; a calibration assistant |
| 4. Decide | @sec-ai-decide | A one-page brief, every number checked | A red team that attacks the chosen policy | Policy search within a budget |

Machine learning appears beside the language models at the stages where it helps: structure discovery in formulation, hybrid models and forecasting contests in testing, and fast surrogate models in policy search.

## Failure gallery

Three ways AI help goes wrong, each caught by a check. The replies below were written for the book with the faults planted.

**A number nobody computed.** We ask for a two-sentence summary of chapter 1's policy result, giving the model the run table.

In [ ]:
runs = {x: looped.run(extra_capacity=x).loc[119] for x in (0, 400)}
table = pd.DataFrame(runs).T[["Waiting list", "Waiting time",
                              "Treatments"]].round(1)
SUMMARY = (f"Summarise this result for a hospital manager in two "
           f"sentences. Rows are extra treatments a month; values are at "
           f"month 119.\n\n{table.to_string()}")
summary = ai.ask(SUMMARY, replay=REPLAY)
print(summary)
print("\nthe table: list shorter by",
      f"{table.loc[0, 'Waiting list'] - table.loc[400, 'Waiting list']:,.0f}")
print("numbers not in the table or the prompt:",
      unsupported(summary, [], SUMMARY) or "none")

The summary reads well, and the check finds the number nobody computed: the table shows the list 2,978 shorter, not 9,000. The claim that the capacity "pays for itself" has no number for the check to catch, and nothing in the model knows about money; that sentence needs a person to delete it.

**Agreeing with the question.** A leading question tends to get the answer it leads to, a tendency called **sycophancy**. We ask the same thing with two opposite framings.

In [ ]:
LEADS = ["Adding 400 treatments a month will clear the waiting list, "
         "won't it? Answer yes or no first, then one sentence.",
         "Adding 400 treatments a month will barely change the waiting "
         "list, won't it? Answer yes or no first, then one sentence."]
for q in LEADS:
    reply = ai.ask(f"{q}\n\nModel equations:\n{looped.equations()}",
                   replay=REPLAY)
    print(f"Q: {q[:58]}...\nA: {reply}\n")

Both framings get a yes. The first is wrong for this model, whose referrals respond to waits; the second is right. Either way, the framing, not the equations, chose the answer. The remedy is the one section 4 used: ask a neutral question, ask for a test, and let the simulation answer.

**An agent that skips the tools.** Given the same tool and a task that invites a quick answer, an agent can answer from its general knowledge without calling anything.

In [ ]:
QUICK = ("Quickly: what extra capacity would bring the waiting time at "
         "month 119 below 6 months? One line.")
lazy = ai.agent(QUICK, {"simulate": simulate}, replay=REPLAY)
ai.show(lazy)
print("tool calls:", sum("tool" in s for s in lazy))
print("unsupported numbers:",
      unsupported(lazy[-1]["answer"], lazy, QUICK + simulate.__doc__)
      or "none")
print("what 900 actually gives:", simulate(900))

No tool call, and two unsupported numbers: 900 and 5.5. The check catches both. Rerunning the guess shows it would work, with a wait of 5.44 months, but it buys 225 more treatments a month than the board needs. A plausible guess that happens to meet the target is the hardest failure to notice, which is why the trace check runs on every answer, right or wrong.

## Check it

Four habits keep AI help honest. The cell below runs them on this chapter.

1. **Every record says how it was made**, and when.
2. **Every structured draft passes its checks** (here, the units of flows).
3. **Every number in an agent's answer traces to a tool.**
4. **An agent's answer reruns to the same result.**

In [ ]:
for k, r in ai.records(REPLAY).items():
    assert r.get("how") and r.get("date"), f"record {k} has no how or date"
assert not check_draft(draft), "the draft's units fail"
assert not unsupported(steps[-1]["answer"], steps,
                       TASK + simulate.__doc__), "untraced number"
final = [s for s in steps if s.get("args", {}).get("extra_capacity") == 675]
assert final and final[-1]["result"] == simulate(675), "answer does not rerun"
print(f"{len(ai.records(REPLAY))} records labelled, draft units balance, "
      "agent answer traced and rerun: all pass")

## Running case

This chapter adds no structure to the waiting list. It adds a habit and a file: from here on, every AI step in the running case is recorded in a replay file, with how it was made, and passes a check before anything built on it is kept. The results so far: a language model's draft of chapter 1's variables matched six of our names and lacked the two that describe how GPs respond to waits; a reasoning model's critique showed, and simulation confirmed, that the sign of the capacity advice depends on that response; and an agent found the extra capacity that brings the wait below six months, 675 a month, which code confirmed. Chapter 3 returns to the people who know the system, and @sec-ai-conceptualise brings AI to that stage.

## Tell the decision maker

When AI helped with an analysis, say so plainly and say what was checked. For example: *"Parts of this analysis were drafted with an AI model. Every number in this note comes from the model runs listed in the appendix, and a member of the team checked each step. The main uncertainty is how strongly GPs' referrals respond to waits: if they respond strongly, extra capacity shortens waits but not the list."*

## Going further

- **Live models.** With `LB_AI_LIVE=1` and a key, every new request goes to a live model and is recorded. Run a chapter's prompts several times live and count how often the checks pass: that rate, not one good reply, is how good a prompt is.
- **Other models.** `ai._live` is a few lines; replace it to use another provider or a model on your own machine, and keep the replay format.
- **Native tool use and structured outputs.** Model services offer their own tool-calling and JSON modes. The book's text protocol is deliberately plain so that every step is visible and replayable; the checks stay the same whichever you use.
- **Agents inside models.** Language models can also stand in for the people inside a simulation, as in generative agent-based modelling [@ghaffarzadegan2024]. That is a different use from the one in this book, where the model helps the modeller.

## Mini-project

Pick a question about chapter 1's model that needs a search, for example *the smallest extra capacity that keeps the list below 28,000 at month 119*.

1. Write a tool for it, with a docstring that says exactly what goes in and what comes out.
2. Write the task, run the agent live if you have a key, and record the replies; otherwise write the replies yourself with `ai.author`, labelled as written.
3. Check the answer with `unsupported`, a rerun and a bisection in code.
4. Write two sentences for the board, and a third that says how AI was used and what was checked.

## Further reading

- @akhavan2024 sets out how language models should and should not be used in simulation modelling, with examples.
- @hosseinichimeh2024 builds and tests a bot that turns text into causal loop diagrams.
- @ghaffarzadegan2024 introduces generative agent-based models, where language models act as the agents.
- @yao2022react describes the reason-then-act loop that `ai.agent` follows, and @wei2022cot the step-by-step prompting behind reasoning models.
- @ji2023hallucination surveys hallucination: why fluent models state things that are not so.